In [1]:
import pandas as pd
import re
import nltk

In [6]:
# Load dataset
import pandas as pd

file_path = "UNITENReview.csv"
df = pd.read_csv(file_path, encoding="latin1")

pd.set_option('display.max_colwidth', None)
print(df.head())
print(df.columns)

                     Timestamp  \
0  2025/02/10 7:40:54 pm GMT+8   
1  2025/02/10 7:41:00 pm GMT+8   
2  2025/02/10 7:41:19 pm GMT+8   
3  2025/02/10 7:46:40 pm GMT+8   
4  2025/02/10 7:46:43 pm GMT+8   

                                                                                                                                                                                                                                                                                                                                                         Review  
0                                                                                                                                                                                                                                                                                                          Im happy with uniten actually, even the people are W  
1                                                                                      

In [7]:
# Ensure the review column name is "Review"
# (Change nothing else; just standardize column name to follow the PDF steps)
for c in df.columns:
    if str(c).strip().lower() == "review":
        df.rename(columns={c: "Review"}, inplace=True)

print(df.columns)

Index(['Timestamp', 'Review'], dtype='object')


In [8]:
# Lowercase conversion
def convert_to_lowercase(text):
    return str(text).lower()

df["lowercased"] = df["Review"].apply(convert_to_lowercase)

# Display column content without truncation
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width
print(df["lowercased"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [9]:
# Removal of URLs
import re

# remove any URLs that start with "http" or "www" from the text
def remove_urls(text):
    return re.sub(r'http\S+|www\S+', '', text)

df["urls_removed"] = df["lowercased"].apply(remove_urls)

# Display column content without truncation
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width
print(df["urls_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [10]:
# Removal of HTML tags
from bs4 import BeautifulSoup

def remove_html(text):
    return BeautifulSoup(text, "html.parser").get_text()

df["html_removed"] = df["urls_removed"].apply(remove_html)

# Display column content without truncation
pd.set_option('display.max_colwidth', None)  # Set to None for unlimited width
print(df["html_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [24]:
# Remove emojis
import emoji

def remove_emojis(text):
    return emoji.replace_emoji(text, replace="")

df["emojis_removed"] = df["html_removed"].apply(remove_emojis)

pd.set_option('display.max_colwidth', None)
print(df["emojis_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [26]:
# Remove emojis again (after encoding fix)
def remove_emojis(text):
    return emoji.replace_emoji(text, replace="")

df["emojis_removed_2"] = df["encoding_fixed"].apply(remove_emojis)

pd.set_option('display.max_colwidth', None)
print(df["emojis_removed_2"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [29]:
# Replace internet slang/chat words
import re

slang_dict = {
    "tbh": "to be honest",
    "omg": "oh my god",
    "lol": "laugh out loud",
    "idk": "I don't know",
    "brb": "be right back",
    "btw": "by the way",
    "imo": "in my opinion",
    "smh": "shaking my head",
    "fyi": "for your information",
    "np": "no problem",
    "ikr": "I know right",
    "asap": "as soon as possible",
    "bff": "best friend forever",
    "gg": "good game",
    "hmu": "hit me up",
    "rofl": "rolling on the floor laughing"
}

def replace_slang(text):

    escaped_slang_words = []

    for word in slang_dict.keys():
        escaped_word = re.escape(word)
        escaped_slang_words.append(escaped_word)

    slang_pattern = r'\b(' + '|'.join(escaped_slang_words) + r')\b'

    def replace_match(match):
        slang_word = match.group(0)
        return slang_dict[slang_word.lower()]

    replaced_text = re.sub(slang_pattern, replace_match, text, flags=re.IGNORECASE)
    return replaced_text

df["slangs_replaced"] = df["emojis_removed_2"].apply(replace_slang)

pd.set_option('display.max_colwidth', None)
print(df["slangs_replaced"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [30]:
# Replace Contractions
contractions_dict = {
    "wasn't": "was not",
    "isn't": "is not",
    "aren't": "are not",
    "weren't": "were not",
    "doesn't": "does not",
    "don't": "do not",
    "didn't": "did not",
    "can't": "cannot",
    "couldn't": "could not",
    "shouldn't": "should not",
    "wouldn't": "would not",
    "won't": "will not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "i'm": "i am",
    "you're": "you are",
    "it's": "it is",
    "i've": "i have"
}

escaped_contractions = []

for contraction in contractions_dict.keys():
    escaped_contraction = re.escape(contraction)
    escaped_contractions.append(escaped_contraction)

joined_contractions = "|".join(escaped_contractions)
contractions_pattern = r'\b(' + joined_contractions + r')\b'
compiled_pattern = re.compile(contractions_pattern, flags=re.IGNORECASE)

def replace_contractions(text):

    def replace_match(match):
        matched_word = match.group(0)
        lower_matched_word = matched_word.lower()
        expanded_form = contractions_dict[lower_matched_word]
        return expanded_form

    expanded_text = compiled_pattern.sub(replace_match, text)
    return expanded_text

df["contractions_replaced"] = df["slangs_replaced"].apply(replace_contractions)

pd.set_option('display.max_colwidth', None)
print(df["contractions_replaced"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [31]:
# Remove punctuations and special characters (KEEP apostrophes)
import string

def remove_punctuations(text):
    punctuation_without_apostrophe = string.punctuation.replace("'", "")
    return text.translate(str.maketrans('', '', punctuation_without_apostrophe))

df["punctuations_removed"] = df["contractions_replaced"].apply(remove_punctuations)

pd.set_option('display.max_colwidth', None)
print(df["punctuations_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [36]:
# Skip spelling correction (it breaks proper nouns like UNITEN)
df["spelling_corrected"] = df["punctuations_removed"]

pd.set_option('display.max_colwidth', None)
print(df["spelling_corrected"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [37]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    filtered_words = []

    for word in words:
        lower_word = word.lower()
        if lower_word not in stop_words:
            filtered_words.append(word)

    return " ".join(filtered_words)

df["stopwords_removed"] = df["spelling_corrected"].apply(remove_stopwords)

pd.set_option('display.max_colwidth', None)
print(df["stopwords_removed"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\affan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [38]:
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(nltk_tag):
    if nltk_tag.startswith('J'):
        return wordnet.ADJ
    elif nltk_tag.startswith('V'):
        return wordnet.VERB
    elif nltk_tag.startswith('N'):
        return wordnet.NOUN
    elif nltk_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lemmatize_text(text):
    if not isinstance(text, str):
        return ""

    words = word_tokenize(text)
    pos_tags = pos_tag(words)

    lemmatized_words = [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]
    return " ".join(lemmatized_words)

df["lemmatized"] = df["stopwords_removed"].apply(lemmatize_text)

pd.set_option('display.max_colwidth', None)
print(df["lemmatized"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              im happy uniten actually 

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\affan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\affan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\affan\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\affan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [40]:
def remove_single_letters(text):
    words = text.split()
    words = [w for w in words if len(w) > 1]
    return " ".join(words)

df["final_clean"] = df["lemmatized"].apply(remove_single_letters)

pd.set_option('display.max_colwidth', None)
print(df["final_clean"])

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                im happy uniten actuall

In [41]:
df.to_csv("UNITENReview_final_clean.csv", index=False)
print("Saved successfully")

Saved successfully
